# Bessel Envelope Extrema Identification

## Purpose

Identify the maxima and minima of the Bessel (jinc) envelope in the fringe data across representative frequency channels, yielding a geometric prior on the solar angular radius $R$ that is immune to amplitude distortions.

## Theory foundation

The AY 121 lab manual ([`interf.tex`](../../../src/ugradio/lab_interf/interf.tex), eq. `interfeqn`) derives the interferometer response to an extended source as the product of the point-source fringe and a **modulating function** — the Fourier (cosine) transform of the source brightness distribution (the one-dimensional form of the van Cittert-Zernike theorem; see theory notebook §1.4 and the astrobaki "Basic Interferometry II" page for the general 2-D statement $V(u,v) = \iint I(l,m)\,e^{-2\pi i(ul+vm)}\,dl\,dm$).

For a uniformly bright circular disk of radius $R$, the lab manual (§8, "Measuring the Diameter of a Circular Source") writes the modulating function as the integral

$$\mathrm{MF}_{\mathrm{theory}} = \frac{1}{R}\int_{-R}^{R}\sqrt{R^2 - \Delta h^2}\;\cos(2\pi f_f \Delta h)\,d\Delta h,$$

and notes that evaluating this analytically "you end up with a Bessel function." The result is the **jinc** (see theory notebook §1.5 for the explicit derivation via the Fourier slice / Hankel-transform identity):

$$\frac{V(q)}{V(0)} = \frac{2\,J_1(2\pi q R)}{2\pi q R},$$

where $q = |u_{\mathrm{sky}}|$ is the projected baseline in wavelengths. This envelope modulates the fringe amplitude: far from transit $q$ is small and $V/V(0) \to 1$; near transit $q$ is large and the visibility oscillates through successive Bessel nulls and sidelobes.

## Strategy

The lab manual's §8 procedure recovers $R$ from the **zero-crossing positions** of this modulating function: since the MF depends only on the product $f_f R$ (equivalently $q R$), the null locations $q_k R = j_{1,k}/(2\pi)$ directly give $R$ once $q_k$ is measured. This notebook generalises that idea to use **all envelope extrema** — both nulls and sidelobe peaks — roughly doubling the number of independent constraints:

1. Load data and apply DC / chip-gain corrections (same pipeline as nb 05).
2. Interpolate $|V|$ onto a uniform HA grid and smooth.
3. Use `find_peaks` to locate minima ($J_1$ zeros) and maxima ($J_2$ zeros) by prominence.
4. Derive a prior $R$ from extrema positions, then export for the full envelope fit in Part III below.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path("..").resolve()))
from utils import (
    F_S_HZ, N_FFT, F_RF0_HZ, PLOT_BAND_GHZ,
    C_LIGHT_MS, OMEGA_EARTH_RAD_S, SIDEREAL_DAY_S,
    NCH_LAT_DEG, NCH_LON_DEG,
    NOMINAL_B_EW_M, NOMINAL_B_NS_M, BAD_CHANNELS,
    SOLAR_DIAMETER_ARCMIN_NOMINAL,
    load_processed_sun_chip_series,
    sky_baseline_lambda,
    adaptive_real_dc_correction,
    apply_chip_gain_correction, CHIP_GAIN_RATIOS_DEFAULT,
    TEXTWIDTH_IN, LABEL_SIZE, TICK_SIZE, LEGEND_SIZE,
    SS_FINE, LW_LIGHT, LW_STANDARD, MS_FINE,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "white"})
print("Imports OK.")

## 1. Load data

In [ ]:
# ── Load chip data ──────────────────────────────────────────────────
DATA_DIR = Path("../../../data/lab03/sun_calibration/chips")
chip_paths = sorted(DATA_DIR.glob("sun_calibration_chip_*.npz"))
print(f"Found {len(chip_paths)} chip files:")
for p in chip_paths:
    print(f"  {p.name}")

chip_data = load_processed_sun_chip_series(chip_paths)

# ── Unpack ──────────────────────────────────────────────────────────
captures    = chip_data.captures
corr_raw    = captures.corr
corr_std    = captures.corr_std
F_SKY_GHZ   = chip_data.f_sky_ghz
F_SKY_HZ    = F_SKY_GHZ * 1e9
DF_HZ       = chip_data.df_hz
unix_mid    = captures.unix_mid
ha_deg_arr  = chip_data.ha_deg
ha_rad_arr  = np.deg2rad(ha_deg_arr)
sun_dec     = chip_data.sun_dec_deg
dec_rad_mean = np.deg2rad(np.nanmean(sun_dec))
dec_rad_arr  = np.deg2rad(sun_dec)
chip_slices  = chip_data.chip_slices
N_chips      = len(chip_slices)

band_mask      = (F_SKY_GHZ >= PLOT_BAND_GHZ[0]) & (F_SKY_GHZ <= PLOT_BAND_GHZ[1])
band_center_hz = np.mean(F_SKY_HZ[band_mask])

print(f"Captures: {len(unix_mid)}, Channels: {len(F_SKY_GHZ)}")
print(f"Analysis band: {PLOT_BAND_GHZ[0]:.3f}–{PLOT_BAND_GHZ[1]:.3f} GHz ({band_mask.sum()} ch)")
print(f"HA range: {ha_deg_arr.min():.1f} to {ha_deg_arr.max():.1f} deg")

In [ ]:
# ── DC correction ───────────────────────────────────────────────────
DC_N_PERIODS       = 3.0
DC_MIN_WINDOW_CAPS = 7
DC_MAX_WINDOW_CAPS = 201

dc_result = adaptive_real_dc_correction(
    corr_chips=[corr_raw[sl] for sl in chip_slices],
    unix_chips=[unix_mid[sl] for sl in chip_slices],
    ha_rad_chips=[ha_rad_arr[sl] for sl in chip_slices],
    bad_channels=BAD_CHANNELS,
    b_ew=NOMINAL_B_EW_M,
    freq_hz=band_center_hz,
    dec_rad=dec_rad_mean,
    n_periods=DC_N_PERIODS,
    min_window_caps=DC_MIN_WINDOW_CAPS,
    max_window_caps=DC_MAX_WINDOW_CAPS,
)
corr_dc = dc_result.corr_dc

# ── Chip-gain correction ───────────────────────────────────────────
import pickle as _pkl

_sidecar = Path("_chip_gain_measured.pkl")
if _sidecar.exists():
    with open(_sidecar, "rb") as _f:
        _cal = _pkl.load(_f)
    chip_gains = np.array(_cal["chip_gains_measured"])
    print(f"Loaded chip gains: {chip_gains}")
else:
    chip_gains = np.array(CHIP_GAIN_RATIOS_DEFAULT[:N_chips])
    print(f"[warning] using default chip gains: {chip_gains}")

corr_dc = apply_chip_gain_correction(corr_dc, chip_slices, chip_gains)

# ── DC-correction alpha ────────────────────────────────────────────
_dc_sidecar = Path("_dc_correction_alpha.pkl")
if _dc_sidecar.exists():
    with open(_dc_sidecar, "rb") as _f:
        DC_ALPHA = float(_pkl.load(_f)["alpha"])
    if abs(DC_ALPHA - 1.0) > 1e-6:
        corr_dc = corr_dc / DC_ALPHA
        print(f"Applied DC alpha = {DC_ALPHA:.4f}")
else:
    print("[warning] DC alpha sidecar not found, assuming 1.0")

# ── Load adopted baseline ──────────────────────────────────────────
import pickle

with open("_baseline_adopted.pkl", "rb") as f:
    _bl = pickle.load(f)
B_EW     = _bl["b_ew_m"]
B_EW_ERR = _bl["b_ew_err_m"]
B_NS     = _bl["b_ns_m"]
B_NS_ERR = _bl["b_ns_err_m"]
print(f"Baseline: B_EW = {B_EW:.4f} m, B_NS = {B_NS:.4f} m")

# ── Good channels & representative indices ─────────────────────────
good_ch = band_mask & ~np.isin(np.arange(corr_dc.shape[1]), list(BAD_CHANNELS))
band_indices = np.where(good_ch)[0]
k_mid = band_indices[len(band_indices) // 2]
print(f"Representative channel: {k_mid} ({F_SKY_GHZ[k_mid]:.3f} GHz)")
print(f"Good channels in band: {good_ch.sum()}")

## 2. Interpolate to uniform HA grid

Interpolate $|V|$ onto a uniform HA grid and smooth with a running mean for peak-finding.

In [ ]:
# Use all good channels in the analysis band
all_ch = np.where(good_ch)[0]
print(f"{len(all_ch)} good channels in analysis band")

# Single representative channel for visualisation
k_mid = all_ch[len(all_ch) // 2]
print(f"Representative channel: ch {k_mid} ({F_SKY_GHZ[k_mid]:.3f} GHz)")

In [ ]:
N_UNIFORM  = 2000    # interpolation grid size
SMOOTH_W   = 21      # running-mean smoothing kernel width

def interpolate_channel(k_ch):
    """Return (ha_uni, mf_smooth, V0, u_uni, env_raw) for channel k_ch."""
    env = np.abs(corr_dc[:, k_ch])
    u = np.abs(sky_baseline_lambda(ha_rad_arr, dec_rad_mean, B_EW, B_NS, float(F_SKY_HZ[k_ch])))

    finite = np.isfinite(ha_deg_arr) & np.isfinite(env)
    order = np.argsort(ha_deg_arr[finite])
    ha_s  = ha_deg_arr[finite][order]
    env_s = env[finite][order]
    u_s   = u[finite][order]

    ha_uni  = np.linspace(ha_s[0], ha_s[-1], N_UNIFORM)
    env_uni = np.interp(ha_uni, ha_s, env_s)
    u_uni   = np.interp(ha_uni, ha_s, u_s)

    V0 = float(np.nanmax(env_uni))
    mf = env_uni / V0

    kernel = np.ones(SMOOTH_W) / SMOOTH_W
    mf_smooth = np.convolve(mf, kernel, mode="same")

    return ha_uni, mf_smooth, V0, u_uni, env

# Process all good channels
ch_data = {}
for k in all_ch:
    ha_uni, mf_sm, V0, u_uni, env_raw = interpolate_channel(k)
    ch_data[k] = dict(ha_uni=ha_uni, mf_smooth=mf_sm, V0=V0, u_uni=u_uni, env_raw=env_raw)

print(f"Processed {len(ch_data)} channels.")

## 3. Find maxima and minima of the Bessel envelope

### Connection to the lab manual's zero-crossing method

The lab manual ([`interf.tex`](../../../src/ugradio/lab_interf/interf.tex), §8) prescribes measuring the solar diameter from the **zero-crossing positions** of the observed modulating function. The logic is clean: the MF depends only on the product $f_f R$, and its zeros occur at $f_f R = j_{1,k}/(2\pi)$ — fixed Bessel constants that are independent of the source amplitude, antenna gains, or system temperature. Measuring the baseline $q$ at each observed zero directly yields $R = j_{1,k}/(2\pi q_k)$.

This notebook extends that logic from nulls alone to **all envelope extrema**. The justification is that the extremum positions are equally amplitude-independent:

- **Minima** (nulls) occur where $J_1(x) = 0$, i.e. $x = 2\pi q R = j_{1,n}$, so $R = j_{1,n} / (2\pi q)$.
- **Maxima** (sidelobe peaks) occur where $\frac{d}{dx}[J_1(x)/x] = 0$. Using the identity $\frac{d}{dx}[J_1(x)/x] = J_0(x)/x - 2J_1(x)/x^2$ and the Bessel recurrence $J_2(x) = 2J_1(x)/x - J_0(x)$, this condition reduces to $J_2(x) = 0$, i.e. $x = j_{2,n}$, so $R = j_{2,n} / (2\pi q)$.

(The $J_2$-zero result for sidelobe peaks is derived in theory notebook §1.7.)

### Why not fit the jinc directly?

The uniform-disk jinc model predicts that the visibility drops to **zero** at the $J_1$ nulls. In practice, the observed $|V|$ at the nulls is significantly inflated above zero due to:

- **Sunspots** — compact bright features add a baseline visibility floor that fills in the nulls. By the linearity of the van Cittert-Zernike theorem ([`interf.tex`](../../../src/ugradio/lab_interf/interf.tex) eq. `interfeqn`; theory notebook §1.7), an unresolved point source at angular offset $\Delta\alpha$ contributes $V_{\mathrm{spot}} \propto e^{-2\pi i u \Delta\alpha}$ — a constant-amplitude visibility that does not vanish at the disk nulls.
- **Limb brightening** — at cm wavelengths, the chromospheric temperature gradient makes the solar limb brighter than the disk centre ([Dulk85] §III; [Selhorst04]; see notebook 05 §7.3 for the quantitative model). This modifies the brightness profile from the uniform $I(\rho) = I_0$ assumed in the lab manual to $I(\rho) = I_0[1 + \varepsilon(\rho/R)^2]$, whose Hankel-transform visibility does not vanish at the jinc nulls (the $\int \rho^3 J_0(2\pi q\rho)\,d\rho$ term is non-zero there).

These effects distort the envelope shape — particularly the null depths — making a direct least-squares fit of $A \cdot |2J_1(x)/x|$ unreliable: the fit would try to compromise between matching the peaks and the non-zero nulls, biasing $R$ and $A$.

Instead, we identify the **positions** of the maxima and minima in HA, which depend on $R$ but **not** on the amplitude distortion. The Bessel roots (where extrema occur in $x = 2\pi q R$) are fixed mathematical constants regardless of sunspots or limb brightening. This gives a robust $R$ estimate per frequency channel.

### Identified features

Since $q$ increases toward transit, **low-order features appear far from transit** and **high-order features appear near transit**. The search windows are computed dynamically from the actual baseline and $D \in [32', 34']$:

| # | Feature | Type | Bessel root | Value |
|---|---------|------|-------------|-------|
| 1 | j$_{2,1}$ | max | 1st zero of $J_2$ | 5.136 |
| 2 | j$_{1,2}$ | min | 2nd zero of $J_1$ | 7.016 |
| 3 | j$_{2,2}$ | max | 2nd zero of $J_2$ | 8.417 |
| 4 | j$_{1,3}$ | min | 3rd zero of $J_1$ | 10.174 |
| 5 | j$_{2,3}$ | max | 3rd zero of $J_2$ | 11.620 |
| 6 | j$_{1,4}$ | min | 4th zero of $J_1$ | 13.324 |
| 7 | j$_{2,4}$ | max | 4th zero of $J_2$ | 14.796 |

### Estimating $R$ from each feature

For any identified extremum at projected baseline $q$:

$$R = \frac{j_{n}}{2\pi\,q}$$

Each feature gives an independent $R$ estimate per frequency channel; the IVW mean across all matched features yields $R_{\mathrm{prior}}$.

In [ ]:
from scipy.signal import find_peaks, peak_prominences
from scipy.special import jn_zeros

# Bessel roots
j1_roots = jn_zeros(1, 8)   # J1 zeros (minima of |V|)
j2_roots = jn_zeros(2, 8)   # J2 zeros (maxima of |V|)

# Features ordered from smallest to largest root (= farthest to nearest transit),
# because sky_baseline_lambda returns the full projected baseline q which is
# proportional to |cos(h)| — INCREASING toward transit.
FEATURES = [
    ("max", j2_roots[0], "j2,1"),  # 1st J2 zero — farthest from transit
    ("min", j1_roots[1], "j1,2"),  # 2nd J1 zero
    ("max", j2_roots[1], "j2,2"),  # 2nd J2 zero
    ("min", j1_roots[2], "j1,3"),  # 3rd J1 zero
    ("max", j2_roots[2], "j2,3"),  # 3rd J2 zero
    # ("min", j1_roots[3], "j1,4"),  # 4th J1 zero
    # ("max", j2_roots[3], "j2,4"),  # 4th J2 zero — nearest transit
]

# Diameter range for window computation
D_LO_ARCMIN = 32.0
D_HI_ARCMIN = 34.5
MARGIN_DEG   = 1.0

R_lo = np.deg2rad(D_LO_ARCMIN / 2.0 / 60.0)
R_hi = np.deg2rad(D_HI_ARCMIN / 2.0 / 60.0)

def u_to_ha(u_target, freq_hz=band_center_hz):
    """Map a target |q| to HA (degrees, negative side) using actual baseline.

    Sweeps HA from -90 to 0 and finds where sky_baseline_lambda = u_target.
    """
    ha_test = np.linspace(-np.pi/2, 0, 10000)
    q_test = sky_baseline_lambda(ha_test, dec_rad_mean, B_EW, B_NS, freq_hz)
    idx = np.argmin(np.abs(q_test - u_target))
    return np.rad2deg(ha_test[idx])

# Compute windows
FEATURE_WINDOWS = []
print(f"Computing HA windows for D = [{D_LO_ARCMIN}, {D_HI_ARCMIN}] arcmin")
print(f"Baseline: B_EW = {B_EW:.4f} m, B_NS = {B_NS:.4f} m")
print(f"Dec: {np.rad2deg(dec_rad_mean):.3f} deg")
print(f"Band centre: {band_center_hz/1e9:.4f} GHz")
print()
print(f"{'Feature':>6} {'Type':>4} {'Root':>8} {'q D=lo':>10} {'q D=hi':>10} "
      f"{'HA D=lo':>9} {'HA D=hi':>9} {'Window':>16}")
print("-" * 82)

for kind, root, label in FEATURES:
    # Smaller R (D_LO) -> larger q needed -> closer to transit (less negative HA)
    # Larger R (D_HI) -> smaller q needed -> farther from transit (more negative HA)
    q_at_lo = root / (2 * np.pi * R_lo)  # larger q
    q_at_hi = root / (2 * np.pi * R_hi)  # smaller q

    ha_at_lo = u_to_ha(q_at_lo)  # less negative (closer to transit)
    ha_at_hi = u_to_ha(q_at_hi)  # more negative (farther from transit)

    # ha_at_hi is more negative, ha_at_lo is less negative
    ha_lo = ha_at_hi - MARGIN_DEG  # extend toward more negative
    ha_hi = ha_at_lo + MARGIN_DEG  # extend toward transit

    FEATURE_WINDOWS.append((ha_lo, ha_hi, kind, root, label))

    print(f"{label:>6} {kind:>4} {root:>8.3f} {q_at_lo:>10.1f} {q_at_hi:>10.1f} "
          f"{ha_at_lo:>+9.1f} {ha_at_hi:>+9.1f} "
          f"[{ha_lo:+.1f}, {ha_hi:+.1f}]")

# Check for overlaps
print()
for i in range(len(FEATURE_WINDOWS) - 1):
    gap = FEATURE_WINDOWS[i+1][0] - FEATURE_WINDOWS[i][1]
    status = "OK" if gap > 0 else "OVERLAP"
    print(f"  Gap {FEATURE_WINDOWS[i][4]} -> {FEATURE_WINDOWS[i+1][4]}: {gap:+.1f} deg  {status}")

In [ ]:
def find_extrema_in_windows(ha_uni, mf_smooth, u_uni, windows=FEATURE_WINDOWS):
    """Find the single most prominent max or min within each HA window.

    Returns a list of dicts with keys: ha, mf, u, type, root, label.
    """
    found = []
    for ha_lo, ha_hi, kind, root, label in windows:
        mask = (ha_uni >= ha_lo) & (ha_uni <= ha_hi)
        if mask.sum() < 3:
            continue
        sub_mf = mf_smooth[mask]
        sub_ha = ha_uni[mask]
        sub_u  = u_uni[mask]

        if kind == "max":
            pk, _ = find_peaks(sub_mf)
            if len(pk) == 0:
                continue
            prom = peak_prominences(sub_mf, pk)[0]
            best = pk[np.argmax(prom)]
        else:
            pk, _ = find_peaks(-sub_mf)
            if len(pk) == 0:
                continue
            prom = peak_prominences(-sub_mf, pk)[0]
            best = pk[np.argmax(prom)]

        found.append({
            "ha": sub_ha[best],
            "mf": sub_mf[best],
            "u": sub_u[best],
            "type": kind,
            "root": root,
            "label": label,
        })
    return found

# Run on ALL good channels
ch_extrema = {}
for k in all_ch:
    d = ch_data[k]
    ch_extrema[k] = find_extrema_in_windows(d["ha_uni"], d["mf_smooth"], d["u_uni"])

# Print summary for mid channel
# k_mid already set in cell 6
print(f"Found extrema for {len(ch_extrema)} channels.")
print(f"Example: ch {k_mid} ({F_SKY_GHZ[k_mid]:.3f} GHz) — {len(ch_extrema[k_mid])} features:")
for f in ch_extrema[k_mid]:
    R_est = f["root"] / (2 * np.pi * f["u"])
    D_est = np.rad2deg(2 * R_est) * 60
    print(f"  {f['label']:>5} {f['type']:>3}: HA = {f['ha']:+7.2f},  "
          f"|u| = {f['u']:.0f},  root = {f['root']:.3f},  D = {D_est:.2f}\'")

## 4. Visualisation

Raw $|V|$ scatter vs hour angle with the identified maxima and minima overlaid (mid-band channel).

In [ ]:
k_show = k_mid
d = ch_data[k_show]
features = ch_extrema[k_show]

fig, ax = plt.subplots(figsize=(TEXTWIDTH_IN, 3.2))

finite = np.isfinite(ha_deg_arr) & np.isfinite(d["env_raw"])
ax.scatter(ha_deg_arr[finite], d["env_raw"][finite] / d["V0"],
           s=0.3, alpha=0.08, color="C0", rasterized=True)

plotted_min, plotted_max = False, False
for f in features:
    if f["type"] == "min":
        ax.axvline(f["ha"], color="C3", lw=LW_LIGHT, ls="--", alpha=0.7,
                   label="minima" if not plotted_min else None)
        ax.plot(f["ha"], f["mf"], "v", ms=7, color="C3", zorder=5)
        plotted_min = True
    else:
        ax.axvline(f["ha"], color="C2", lw=LW_LIGHT, ls=":", alpha=0.7,
                   label="maxima" if not plotted_max else None)
        ax.plot(f["ha"], f["mf"], "^", ms=7, color="C2", zorder=5)
        plotted_max = True

# Show window boundaries
for ha_lo, ha_hi, kind, root, label in FEATURE_WINDOWS:
    ax.axvspan(ha_lo, ha_hi, alpha=0.03, color="C2" if kind == "max" else "C3")

ax.set_xlabel("Hour angle [deg]")
ax.set_ylabel("Normalised $|V|$")
ax.set_title(f"ch {k_show} ({F_SKY_GHZ[k_show]:.3f} GHz)", fontsize=TICK_SIZE)
ax.legend(fontsize=TICK_SIZE - 1, loc="upper right")
ax.set_ylim(-0.05, 1.15)
fig.tight_layout()
plt.show()

## 5. Prior $R$ from extrema positions

Each identified extremum gives an independent solar radius estimate: $R = j_k / (2\pi\, |u|_{\mathrm{measured}})$. Average across all matched extrema to obtain $R_{\mathrm{prior}}$ per channel.

In [ ]:
# Compute R_prior for each good channel from window-matched features
R_prior_per_ch = {}
for k in all_ch:
    features = ch_extrema[k]
    if len(features) < 2:
        continue

    R_estimates = [f["root"] / (2 * np.pi * f["u"]) for f in features]

    R_prior_per_ch[k] = {
        "R_mean": float(np.mean(R_estimates)),
        "R_std": float(np.std(R_estimates, ddof=1)),
        "n": len(R_estimates),
    }

# ── IVW estimate across all channels ───────────────────────────────
prior_keys = sorted(R_prior_per_ch.keys())
prior_freqs = np.array([F_SKY_GHZ[k] for k in prior_keys])
prior_D = np.array([np.rad2deg(2 * R_prior_per_ch[k]["R_mean"]) * 60 for k in prior_keys])
prior_D_err = np.array([np.rad2deg(2 * R_prior_per_ch[k]["R_std"]) * 60 for k in prior_keys])

valid = prior_D_err > 0.01
w = 1.0 / prior_D_err[valid]**2
D_ivw = float(np.sum(w * prior_D[valid]) / np.sum(w))
D_ivw_err = float(1.0 / np.sqrt(np.sum(w)))
R_prior_ivw = np.deg2rad(D_ivw / 2.0 / 60.0)

print(f"Channels with valid R_prior: {valid.sum()} / {len(all_ch)}")
print(f"IVW prior diameter: {D_ivw:.3f} +/- {D_ivw_err:.3f} arcmin")
print(f"IVW prior radius:   {np.rad2deg(R_prior_ivw)*60:.3f} arcmin")

# ── Plot D_prior vs frequency ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(TEXTWIDTH_IN, 3.0))
ax.errorbar(prior_freqs[valid], prior_D[valid], yerr=prior_D_err[valid],
            fmt=".", ms=3, color="C0", alpha=0.4, elinewidth=0.5, capsize=0,
            label="per-channel prior")
ax.axhline(D_ivw, color="C1", lw=LW_STANDARD,
           label=f"IVW = {D_ivw:.2f} $\pm$ {D_ivw_err:.2f} arcmin")
ax.axhline(SOLAR_DIAMETER_ARCMIN_NOMINAL, color="0.4", ls="--", lw=LW_LIGHT,
           label=f"nominal {SOLAR_DIAMETER_ARCMIN_NOMINAL}\'")
ax.set_xlabel("Sky frequency [GHz]")
ax.set_ylabel("Solar diameter [arcmin]")
ax.set_title("Step 1: Prior diameter from extrema positions", fontsize=TICK_SIZE)
ax.legend(fontsize=TICK_SIZE - 1)
fig.tight_layout()
plt.show()

# ── Export for downstream notebooks ────────────────────────────────
_r_prior_sidecar = Path("_R_prior_per_ch.pkl")
import pickle as _pkl
with open(_r_prior_sidecar, "wb") as _f:
    _pkl.dump({
        "R_prior_per_ch": R_prior_per_ch,
        "D_ivw": D_ivw,
        "D_ivw_err": D_ivw_err,
        "R_prior_ivw": R_prior_ivw,
    }, _f)
print(f"Exported R_prior to {_r_prior_sidecar}")

In [ ]:
# Identify outlier channels with prior D < 33.9 arcmin
D_CUT = 33.9
outlier_keys = [k for k in prior_keys
                if np.rad2deg(2 * R_prior_per_ch[k]["R_mean"]) * 60 < D_CUT]
print(f"Outlier channels (D_prior < {D_CUT}\'): {len(outlier_keys)}")
for k in outlier_keys:
    D_k = np.rad2deg(2 * R_prior_per_ch[k]["R_mean"]) * 60
    print(f"  ch {k}: {F_SKY_GHZ[k]:.3f} GHz, D = {D_k:.2f}\'")

## 6. Prior jinc envelope vs data

Plot the jinc envelope from $R_{\mathrm{prior}}$ (IVW) against the raw $|V|$ for the representative channel, with a fill-between band showing the $\pm 1\sigma$ range.

In [ ]:
from scipy.special import j1 as J1

def jinc_envelope(q, A, R):
    """Uniform disk visibility: A * |2 J1(2 pi q R) / (2 pi q R)|."""
    x = 2 * np.pi * q * R
    out = np.ones_like(x, dtype=float)
    nz = x != 0
    out[nz] = np.abs(2 * J1(x[nz]) / x[nz])
    return A * out

k_show = k_mid
d = ch_data[k_show]

# Raw |V| and q for this channel
env_raw = np.abs(corr_dc[:, k_show])
q_raw = sky_baseline_lambda(ha_rad_arr, dec_rad_mean, B_EW, B_NS, float(F_SKY_HZ[k_show]))
finite = np.isfinite(env_raw) & np.isfinite(ha_deg_arr)

# Amplitude estimate: peak of raw data
A_est = float(np.nanmax(env_raw[finite]))

# Fine HA grid for smooth curves
ha_fine = d["ha_uni"]
q_fine  = d["u_uni"]

# IVW prior R and its error
R_center = R_prior_ivw
R_err = np.deg2rad(D_ivw_err / 2.0 / 60.0)

env_center = jinc_envelope(q_fine, A_est, R_center)
env_lo     = jinc_envelope(q_fine, A_est, R_center - R_err)
env_hi     = jinc_envelope(q_fine, A_est, R_center + R_err)
# Envelope of the two curves at each point (fill between min and max)
env_band_lo = np.minimum(env_lo, env_hi)
env_band_hi = np.maximum(env_lo, env_hi)

fig, ax = plt.subplots(figsize=(TEXTWIDTH_IN, 3.5))

# Raw data
ax.scatter(ha_deg_arr[finite], env_raw[finite] / A_est,
           s=0.3, alpha=0.08, color="C0", rasterized=True, label="$|V|$ data")

# Prior jinc
ax.plot(ha_fine, env_center / A_est, lw=LW_STANDARD, color="C1",
        label=f"jinc prior (D = {D_ivw:.1f}\')")

# Error band
ax.fill_between(ha_fine, env_band_lo / A_est, env_band_hi / A_est,
                color="C1", alpha=0.2, label=f"$\pm${D_ivw_err:.2f}\' band")

# Mark extrema
for f in ch_extrema[k_show]:
    if f["type"] == "min":
        ax.axvline(f["ha"], color="C3", lw=LW_LIGHT, ls="--", alpha=0.5)
    else:
        ax.axvline(f["ha"], color="C2", lw=LW_LIGHT, ls=":", alpha=0.5)

ax.set_xlabel("Hour angle [deg]")
ax.set_ylabel("Normalised $|V|$")
ax.set_title(f"ch {k_show} ({F_SKY_GHZ[k_show]:.3f} GHz)", fontsize=TICK_SIZE)
ax.legend(fontsize=TICK_SIZE - 1, loc="upper right")
ax.set_ylim(-0.05, 1.15)
fig.tight_layout()
plt.show()

In [ ]:
# 20 channels whose prior D deviates most from the IVW
N_SHOW = 20
deviation = np.abs(prior_D - D_ivw)
worst_idx = np.argsort(deviation[valid])[::-1][:N_SHOW]
worst_keys = [prior_keys[np.where(valid)[0][i]] for i in worst_idx]

print(f"Top {N_SHOW} channels by |D_prior - D_ivw| (IVW = {D_ivw:.2f}\'):")
for k in worst_keys:
    D_k = np.rad2deg(2 * R_prior_per_ch[k]["R_mean"]) * 60
    print(f"  ch {k} ({F_SKY_GHZ[k]:.3f} GHz): D = {D_k:.2f}\', delta = {D_k - D_ivw:+.2f}\'")

fig, axes = plt.subplots(N_SHOW, 1, figsize=(TEXTWIDTH_IN, 1.8 * N_SHOW),
                         sharex=True, sharey=True)

for i, k in enumerate(worst_keys):
    ax = axes[i]
    d = ch_data[k]
    env = d["env_raw"]
    finite = np.isfinite(ha_deg_arr) & np.isfinite(env)
    ax.scatter(ha_deg_arr[finite], env[finite] / d["V0"],
               s=0.3, alpha=0.15, color="C0", rasterized=True)

    for f in ch_extrema[k]:
        if f["type"] == "min":
            ax.axvline(f["ha"], color="C3", lw=0.5, ls="--", alpha=0.6)
        else:
            ax.axvline(f["ha"], color="C2", lw=0.5, ls=":", alpha=0.6)

    D_k = np.rad2deg(2 * R_prior_per_ch[k]["R_mean"]) * 60
    ax.set_ylabel("$|V|/V_0$", fontsize=TICK_SIZE - 1)
    ax.text(0.98, 0.92, f"ch {k} ({F_SKY_GHZ[k]:.3f} GHz)  D={D_k:.1f}\' ({D_k-D_ivw:+.1f}\')",
            transform=ax.transAxes, ha="right", va="top", fontsize=TICK_SIZE - 2)
    ax.set_ylim(-0.05, 1.15)

axes[-1].set_xlabel("Hour angle [deg]")
fig.suptitle(f"Top {N_SHOW} channels by deviation from IVW ({D_ivw:.1f}\')", fontsize=TICK_SIZE)
fig.tight_layout()
plt.show()

# Solar Diameter Fit: Limb-Brightened Disk + Sunspot Floor

## Purpose

Fit a physically motivated 3-parameter visibility model to the fringe envelope, with the solar radius $R$ fixed per channel from the geometric extrema analysis in Part II below and the baseline fixed from notebook 04.

## Starting point: the measurement equation

The AY 121 lab manual ([`interf.tex`](../../../src/ugradio/lab_interf/interf.tex), eq. `interfeqn`) derives the interferometer response to an extended source as the product of the point-source fringe and the Fourier transform of the source brightness — the one-dimensional van Cittert-Zernike theorem. The astrobaki "Basic Interferometry II" page gives the full 2-D form:

$$V(u,v) = \iint I(l,m)\,e^{-2\pi i(ul + vm)}\,dl\,dm.$$

For a circularly symmetric source, this reduces to a Hankel transform (theory notebook §1.5):

$$V(q) = 2\pi \int_0^R I(\rho)\,J_0(2\pi q\rho)\,\rho\,d\rho,$$

where $q = \sqrt{u^2 + v^2}$ is the projected baseline length in wavelengths.

## Step 1: Uniform disk $\to$ jinc (the lab-manual model)

The lab manual (§8, "Measuring the Diameter of a Circular Source") assumes a uniformly bright disk $I(\rho) = I_0$ and writes the modulating function as the integral $\frac{1}{R}\int_{-R}^{R}\sqrt{R^2-\Delta h^2}\cos(2\pi f_f \Delta h)\,d\Delta h$, noting that it evaluates to a Bessel function. The 2-D Hankel form makes this explicit: using the standard identity $\int_0^R J_0(a\rho)\,\rho\,d\rho = R\,J_1(aR)/a$,

$$V_{\mathrm{disk}}(q) = I_0\,\pi R^2 \cdot \frac{2\,J_1(2\pi q R)}{2\pi q R}.$$

The prefactor $I_0 \pi R^2$ is the total flux; defining $A \equiv I_0 \pi R^2$ so that $V_{\mathrm{disk}}(0) = A$, the normalised visibility is the **jinc**: $V_{\mathrm{disk}}/A = 2J_1(x)/x$ with $x = 2\pi q R$.

## Step 2: Generalise to a limb-brightened disk

The lab manual acknowledges the Sun is not uniformly bright. At centimetre wavelengths the chromospheric temperature increases with height, making the solar limb brighter than the disk centre — the opposite of optical limb darkening ([Dulk85] §III; [Selhorst04]; see notebook 05 §7.3 for observational references). The simplest extension that preserves circular symmetry is a quadratic radial profile:

$$I(\rho) = I_0\!\left[1 + \varepsilon\!\left(\frac{\rho}{R}\right)^{\!2}\right], \qquad \rho \le R,$$

where $\varepsilon > 0$ parametrises the limb excess. Substituting into the Hankel transform:

$$V(q) = 2\pi I_0 \int_0^R \!\left[1 + \varepsilon\!\left(\frac{\rho}{R}\right)^{\!2}\right] J_0(2\pi q\rho)\,\rho\,d\rho
       = \underbrace{2\pi I_0 \int_0^R J_0(2\pi q\rho)\,\rho\,d\rho}_{\text{uniform disk}} \;+\; \underbrace{\frac{2\pi I_0\,\varepsilon}{R^2}\int_0^R \rho^3\,J_0(2\pi q\rho)\,d\rho}_{\text{quadratic (limb) term}}.$$

The first integral is the standard jinc above. For the second, substitute $t = \rho/R$:

$$\int_0^R \rho^3\,J_0(2\pi q\rho)\,d\rho = R^4 \int_0^1 t^3\,J_0(x\,t)\,dt, \qquad x = 2\pi q R.$$

The total flux is $V(0) = I_0\pi R^2(1 + \varepsilon/2)$, so normalising:

$$\boxed{V_{\mathrm{lb}}(q) = \frac{\frac{2\,J_1(x)}{x} \;+\; \varepsilon\cdot 2\!\int_0^1 t^3\,J_0(x\,t)\,dt}{1 + \varepsilon/2},}$$

where the denominator ensures $V_{\mathrm{lb}}(0) = 1$ (using $2J_1(x)/x \to 1$ and $2\int_0^1 t^3\,dt = 1/2$ as $x \to 0$). The integral $\int_0^1 t^3 J_0(xt)\,dt$ is precomputed as a lookup table in the implementation below.

## Step 3: Add an unresolved sunspot

By the linearity of the van Cittert-Zernike theorem ([`interf.tex`](../../../src/ugradio/lab_interf/interf.tex) eq. `interfeqn`; theory notebook §1.7), the visibility of a composite source is the sum of its components' visibilities. An unresolved sunspot at unknown position on the disk contributes a point-source visibility with constant amplitude $f$ (the spot's flux as a fraction of the disk flux) but **unknown phase** $\phi(q)$ set by its angular offset:

$$V_{\mathrm{total}}(q) = A\,V_{\mathrm{lb}}(q) + f\cdot A\,e^{i\phi(q)}.$$

We observe the **amplitude** $|V_{\mathrm{total}}|$. Since we do not know $\phi$, the cross term $2\,V_{\mathrm{lb}}\,f\cos\phi$ is uncontrolled. Adding in quadrature (equivalent to averaging over unknown $\phi$, or treating the spot contribution as incoherent with the disk):

$$|V_{\mathrm{total}}(q)| = A\sqrt{V_{\mathrm{lb}}(q)^2 + f^2}.$$

## Result: the 3-parameter objective function

Combining Steps 1-3:

$$\boxed{|V(h)| = A\sqrt{V_{\mathrm{lb}}(q;\,R,\,\varepsilon)^2 + f^2},}$$

where $q(h)$ is the projected baseline from the adopted $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ and the per-capture geometry (theory notebook §1.3; [`interf.tex`](../../../src/ugradio/lab_interf/interf.tex) eq. `fringefreq`). $R$ is fixed per channel from Part II; the baseline is fixed from 04.

**3 free parameters:** $A$ (overall amplitude), $\varepsilon$ (limb-brightening coefficient), $f$ (sunspot visibility floor).

## 2. Model implementation

The derivation above yields three functions to implement:

1. **Uniform-disk jinc:** $2J_1(x)/x$ — evaluated directly via `scipy.special.j1`.
2. **Quadratic limb term:** $2\int_0^1 t^3 J_0(xt)\,dt$ — precomputed on a fine grid of $x$ via numerical quadrature (`scipy.integrate.quad`) and interpolated with a cubic spline LUT for speed.
3. **Combined normalised visibility:** $V_{\mathrm{lb}}(q) = [2J_1(x)/x + \varepsilon \cdot v_{\mathrm{quad}}(x)] / (1 + \varepsilon/2)$.

The projected baseline $q(h)$ is computed from the adopted $(b_{\mathrm{ew}}, b_{\mathrm{ns}})$ and per-capture geometry via the standard baseline projection ([`interf.tex`](../../../src/ugradio/lab_interf/interf.tex) eq. `nutau`; theory notebook §1.3):

$$q = \sqrt{u^2 + v_0^2}, \qquad u = \frac{b_{\mathrm{ew}}}{\lambda}\cos\delta\cos h - \frac{b_{\mathrm{ns}}}{\lambda}\sin L\cos\delta\sin h, \qquad v_0 = \frac{b_{\mathrm{ns}}}{\lambda}\cos L.$$

In [ ]:
from scipy.special import j0 as J0, j1 as J1
from scipy.integrate import quad
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d

# Precompute limb-brightening Hankel integral on a lookup table
_X_LUT = np.linspace(0, 60, 5000)
_V_QUAD_LUT = np.empty_like(_X_LUT)
for _i, _xi in enumerate(_X_LUT):
    if _xi < 1e-8:
        _V_QUAD_LUT[_i] = 0.5
    else:
        _integrand = lambda t, xi=_xi: t**3 * J0(xi * t)
        _V_QUAD_LUT[_i] = 2.0 * quad(_integrand, 0, 1)[0]
_v_quad_interp = interp1d(_X_LUT, _V_QUAD_LUT, kind='cubic',
                           bounds_error=False, fill_value=0.0)
print(f"LUT precomputed: {len(_X_LUT)} points, x in [0, {_X_LUT[-1]}]")


def limb_brightened_visibility(q_lambda, R_rad, epsilon):
    x = 2.0 * np.pi * np.abs(q_lambda) * R_rad
    v_disk = np.where(x > 1e-8, 2.0 * J1(x) / x, 1.0)
    v_quad = _v_quad_interp(x)
    return (v_disk + epsilon * v_quad) / (1.0 + epsilon / 2.0)


_SIN_LAT = np.sin(np.deg2rad(NCH_LAT_DEG))
_COS_LAT = np.cos(np.deg2rad(NCH_LAT_DEG))


def compute_q(ha_rad, dec_rad, b_ew, b_ns, freq_hz):
    lam = C_LIGHT_MS / freq_hz
    cos_dec = np.cos(dec_rad)
    u = (b_ew / lam) * cos_dec * np.cos(ha_rad) - (b_ns / lam) * _SIN_LAT * cos_dec * np.sin(ha_rad)
    v0 = (b_ns / lam) * _COS_LAT
    return np.sqrt(u**2 + v0**2)


def model_V_abs(q_lambda, A, R, eps, f):
    v_lb = limb_brightened_visibility(q_lambda, R, eps)
    return A * np.sqrt(v_lb**2 + f**2)


# Sanity check
q_test = compute_q(np.linspace(-np.pi/2, 0, 100), dec_rad_mean, B_EW, B_NS, band_center_hz)
print(f"q range: [{q_test.min():.0f}, {q_test.max():.0f}] wavelengths")
print("Model defined.")

## 3. Interpolate and fit per channel

$R$ fixed from Part II. Baseline fixed from 04.

In [ ]:
# ── R_prior already computed above (Part II) ──────────────────
# No need to load from sidecar — R_prior_per_ch, D_ivw, D_ivw_err,
# R_prior_ivw are all in memory from the extrema analysis above.
print(f"Using R_prior from Part II: D_ivw = {D_ivw:.3f} +/- {D_ivw_err:.3f} arcmin")
print(f"R_prior_ivw = {np.rad2deg(R_prior_ivw)*60:.3f} arcmin")
print(f"Channels with per-channel R: {len(R_prior_per_ch)}")


In [ ]:
N_UNIFORM = 2000

def fit_channel(k_ch):
    if k_ch in R_prior_per_ch:
        R_fixed = R_prior_per_ch[k_ch]["R_mean"]
    else:
        R_fixed = R_prior_ivw

    freq_hz = float(F_SKY_HZ[k_ch])
    env = np.abs(corr_dc[:, k_ch])
    q = compute_q(ha_rad_arr, dec_rad_mean, B_EW, B_NS, freq_hz)

    finite = np.isfinite(env) & np.isfinite(ha_deg_arr)
    order = np.argsort(ha_deg_arr[finite])
    ha_sorted  = ha_deg_arr[finite][order]
    env_sorted = env[finite][order]
    q_sorted   = q[finite][order]

    ha_uni  = np.linspace(ha_sorted[0], ha_sorted[-1], N_UNIFORM)
    env_uni = np.interp(ha_uni, ha_sorted, env_sorted)
    q_uni   = np.interp(ha_uni, ha_sorted, q_sorted)

    A0 = float(np.nanmax(env_uni))

    def _model(q_arr, A, eps, f):
        v_lb = limb_brightened_visibility(q_arr, R_fixed, eps)
        return A * np.sqrt(v_lb**2 + f**2)

    try:
        popt, pcov = curve_fit(
            _model, q_uni, env_uni,
            p0=[A0, 0.1, 0.02],
            bounds=(
                [0,      0,   0],
                [A0*3,   2.0, 0.15],
            ),
            maxfev=20000,
        )
        perr = np.sqrt(np.diag(pcov))
        names = ["A", "eps", "f"]
        result = {n: float(popt[i]) for i, n in enumerate(names)}
        result.update({n + "_err": float(perr[i]) for i, n in enumerate(names)})
        result["R"] = R_fixed
        result["ha_uni"] = ha_uni
        result["q_uni"] = q_uni
        result["env_uni"] = env_uni
        return result
    except (RuntimeError, ValueError):
        return None


res = fit_channel(k_mid)
if res is not None:
    D = np.rad2deg(2 * res["R"]) * 60
    print(f"ch {k_mid} ({F_SKY_GHZ[k_mid]:.3f} GHz):")
    print(f"  R = {np.rad2deg(res['R'])*60:.3f} arcmin (fixed from Part II)")
    print(f"  A = {res['A']:.4f} +/- {res['A_err']:.4f}")
    print(f"  eps = {res['eps']:.3f} +/- {res['eps_err']:.3f}")
    print(f"  f = {res['f']:.4f} +/- {res['f_err']:.4f}")
else:
    print(f"ch {k_mid}: fit failed")

## 4. Visualisation: fit vs data (representative channel)

In [ ]:
fr = res

freq_hz = float(F_SKY_HZ[k_mid])
env_raw = np.abs(corr_dc[:, k_mid])
finite = np.isfinite(env_raw) & np.isfinite(ha_deg_arr)

v_lb_fit = limb_brightened_visibility(fr["q_uni"], fr["R"], fr["eps"])
fit_curve = fr["A"] * np.sqrt(v_lb_fit**2 + fr["f"]**2)

V0 = fr["A"]
D_show = np.rad2deg(2 * fr["R"]) * 60

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(TEXTWIDTH_IN, 5.0),
                                      gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
                                      sharex=True)

ax_top.scatter(ha_deg_arr[finite], env_raw[finite] / V0,
               s=0.3, alpha=0.05, color="C0", rasterized=True, label="raw $|V|$")
ax_top.plot(fr["ha_uni"], fr["env_uni"] / V0, lw=0.5, color="C0", alpha=0.4,
            label="interpolated")

fit_label = (f"D={D_show:.1f}' (fixed), "
             rf"$\varepsilon$={fr['eps']:.2f}, "
             f"f={fr['f']:.3f}")
ax_top.plot(fr["ha_uni"], fit_curve / V0, lw=LW_STANDARD, color="C1", label=fit_label)

ax_top.set_ylabel("Normalised $|V|$")
ax_top.set_title(f"ch {k_mid} ({F_SKY_GHZ[k_mid]:.3f} GHz)", fontsize=TICK_SIZE)
ax_top.legend(fontsize=TICK_SIZE - 2, loc="upper right")
ax_top.set_ylim(-0.05, 1.15)
ax_top.tick_params(labelbottom=False)

resid = fr["env_uni"] / V0 - fit_curve / V0
ax_bot.plot(fr["ha_uni"], resid, lw=0.5, color="C0", alpha=0.6)
ax_bot.axhline(0, color="0.4", lw=LW_LIGHT, ls="--")
ax_bot.set_xlabel("Hour angle [deg]")
ax_bot.set_ylabel("Residual")

fig.tight_layout()
plt.show()

## 5. Fit all channels

In [ ]:
print(f"Fitting all {len(all_ch)} channels (3-param, R+baseline fixed)...")
fit_results = {}
for i, k in enumerate(all_ch):
    r = fit_channel(k)
    if r is not None:
        fit_results[k] = {key: r[key] for key in ["A", "eps", "f", "R",
                                                    "A_err", "eps_err", "f_err"]}
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(all_ch)} done, {len(fit_results)} converged")

print(f"\nConverged: {len(fit_results)} / {len(all_ch)} channels")

## 6. Results: fitted parameters vs frequency

In [ ]:
fit_keys = sorted(fit_results.keys())
fit_freqs = np.array([F_SKY_GHZ[k] for k in fit_keys])
fit_D     = np.array([np.rad2deg(2 * fit_results[k]["R"]) * 60 for k in fit_keys])
fit_eps   = np.array([fit_results[k]["eps"] for k in fit_keys])
fit_f     = np.array([fit_results[k]["f"] for k in fit_keys])

fig, axes = plt.subplots(3, 1, figsize=(TEXTWIDTH_IN, 7.5), sharex=True)

ax = axes[0]
ax.plot(fit_freqs, fit_D, ".", ms=3, color="C0", alpha=0.4)
ax.axhline(D_ivw, color="C1", lw=LW_STANDARD,
           label=f"IVW = {D_ivw:.2f}'")
ax.set_ylabel("D [arcmin] (fixed)")
ax.legend(fontsize=TICK_SIZE - 1)

ax = axes[1]
ax.plot(fit_freqs, fit_eps, ".", ms=3, color="C2", alpha=0.4)
ax.axhline(np.median(fit_eps), color="C2", lw=LW_LIGHT,
           label=rf"median $\varepsilon$ = {np.median(fit_eps):.3f}")
ax.set_ylabel(r"$\varepsilon$")
ax.legend(fontsize=TICK_SIZE - 1)

ax = axes[2]
ax.plot(fit_freqs, fit_f, ".", ms=3, color="C3", alpha=0.4)
ax.axhline(np.median(fit_f), color="C3", lw=LW_LIGHT,
           label=f"median f = {np.median(fit_f):.4f}")
ax.set_ylabel("f (spot)")
ax.set_xlabel("Sky frequency [GHz]")
ax.legend(fontsize=TICK_SIZE - 1)

fig.suptitle("3-parameter fit (R + baseline fixed)", fontsize=TICK_SIZE)
fig.tight_layout()
plt.show()

print(f"Median eps: {np.median(fit_eps):.4f}")
print(f"Median f: {np.median(fit_f):.4f}")

## 7. Diameter summary

The adopted solar diameter comes from two complementary measurements:

1. **Geometric prior (Part II):** The IVW diameter from Bessel-extrema positions, $D_{\mathrm{IVW}}$, is purely geometric — it depends only on the *positions* of envelope features, not their amplitudes, and is therefore immune to limb brightening and sunspot contamination.
2. **Envelope fit (Part III):** The 3-parameter fit decomposes the envelope shape into a limb-brightened disk ($\varepsilon$) and a sunspot floor ($f$), with $R$ fixed from Part II.

The geometric prior is the primary diameter result; the envelope fit provides the physical characterisation of the brightness profile.

### Comparison targets

- **Optical photosphere:** $\theta_\odot^{\mathrm{opt}} \approx 31.6'$ — the operational lab value.
- **Radio Sun:** $\theta_\odot(17\,\mathrm{GHz}) = 32.55 \pm 0.05'$ from [Selhorst04], with the 10 GHz value expected to be slightly larger because the longer wavelength samples higher chromospheric layers.


In [ ]:
# --- Diameter summary ---
THETA_RADIO_ARCMIN = 32.55  # Selhorst+04 NoRH 17 GHz
THETA_OPT_ARCMIN   = SOLAR_DIAMETER_ARCMIN_NOMINAL  # 31.6' optical

D_adopted = D_ivw          # geometric prior from Part II
D_err     = D_ivw_err      # IVW error

print("=" * 60)
print("SOLAR DIAMETER RESULT")
print("=" * 60)
print(f"  Geometric prior (Bessel extrema, IVW across {len(R_prior_per_ch)} channels):")
print(f"    D = {D_adopted:.3f} +/- {D_err:.3f} arcmin")
print()
print(f"  Envelope fit (3-param, R fixed from above):")
print(f"    median epsilon = {np.median(fit_eps):.3f} (limb brightening)")
print(f"    median f       = {np.median(fit_f):.4f} (sunspot floor)")
print()
print("Comparison with literature:")
print(f"  Optical reference:     {THETA_OPT_ARCMIN:.1f}' [AY121-Lab3]")
print(f"  Radio 17 GHz:          {THETA_RADIO_ARCMIN:.2f}' [Selhorst+04]")
print(f"  Radio K-band:          32.6-32.7' [Marongiu+24]")
print(f"  10 GHz expected:       ~32.6' (slightly larger than 17 GHz)")
print()

d_opt = (D_adopted - THETA_OPT_ARCMIN) / THETA_OPT_ARCMIN * 100
d_rad = (D_adopted - THETA_RADIO_ARCMIN) / THETA_RADIO_ARCMIN * 100
n_sig_opt = abs(D_adopted - THETA_OPT_ARCMIN) / D_err if D_err > 0 else float("inf")
n_sig_rad = abs(D_adopted - THETA_RADIO_ARCMIN) / D_err if D_err > 0 else float("inf")

print(f"  vs optical:  {d_opt:+.2f}%  ({n_sig_opt:.1f} sigma)")
print(f"  vs radio:    {d_rad:+.2f}%  ({n_sig_rad:.1f} sigma)")
print()
if D_adopted > THETA_OPT_ARCMIN:
    print("The radio diameter exceeds the optical value, consistent with")
    print("the expectation that cm-wavelength emission originates from the")
    print("chromosphere (above the photosphere), producing a larger apparent radius.")
if abs(d_rad) < 5:
    print(f"The result is within {abs(d_rad):.1f}% of the Selhorst+04 17 GHz measurement,")
    print("consistent with the 10 GHz radius being comparable to or slightly larger.")


## 8. Error budget - statistical and systematic contributions

Here we combine every known contribution to the diameter uncertainty into a single table. The dominant terms are:

- **Baseline uncertainty:** $\sigma_{b_{\mathrm{ew}}}/b_{\mathrm{ew}}$ propagates directly into the diameter because the Bessel argument is $2\pi u R$ and $u \propto b_{\mathrm{ew}}$.
- **DC-correction amplitude bias $\alpha$** from the notebook-02b synthetic injection test.
- **Limb brightening position shift:** the extrema-based diameter is insensitive to envelope *amplitude* distortions, but a non-uniform brightness profile can shift the extremum *positions* by a small amount (sub-percent for realistic $\varepsilon$).
- **Processing terms:** chip-gain residual and bandwidth smearing are smaller but still tracked explicitly.

In [ ]:
# --- Error budget table ---
from utils import NOMINAL_B_EW_ERR_M

contributions = []

# 1. Statistical: from IVW of per-channel extrema
contributions.append(("statistical (extrema IVW)", D_ivw_err))

# 2. Baseline uncertainty -> fractional bias on R
frac_b = float(B_EW_ERR) / float(B_EW) if B_EW_ERR > 0 else float(NOMINAL_B_EW_ERR_M) / float(B_EW)
contributions.append(
    ("baseline (sigma_bew / bew)", frac_b * D_adopted)
)

# 3. DC-correction amplitude bias
_dc_sidecar = Path("_dc_correction_alpha.pkl")
if _dc_sidecar.exists():
    import pickle as _pkl2
    with open(_dc_sidecar, "rb") as _f2:
        _dc_alpha_val = float(_pkl2.load(_f2)["alpha"])
else:
    _dc_alpha_val = 1.0
frac_dc = abs(_dc_alpha_val - 1.0)
contributions.append(
    ("DC-correction alpha", frac_dc * D_adopted)
)

# 4. Chip-gain residual (~10% of the chip-to-chip step)
chip_gain_step = float(np.max(np.abs(chip_gains - 1.0)))
frac_cg = 0.1 * chip_gain_step  # residual after correction
contributions.append(
    ("chip-gain residual", frac_cg * D_adopted)
)

# 5. Limb brightening — now measured directly via epsilon
#    (included for comparison; the extrema method is insensitive to this)
frac_lb = 0.005  # extrema positions shift < 0.5% even for eps ~ 2
contributions.append(
    ("limb bright. (extrema position shift)", frac_lb * D_adopted)
)

# 6. Bandwidth smearing
frac_bw = 0.003
contributions.append(
    ("bandwidth smearing", frac_bw * D_adopted)
)

print(f"{'contribution':<40} {'sigma [arcmin]':>18}")
print("-" * 60)
total_sq = 0.0
for label, val in contributions:
    print(f"{label:<40} {val:>18.4f}")
    total_sq += val ** 2
total = float(np.sqrt(total_sq))
print("-" * 60)
print(f"{'TOTAL (quadrature sum)':<40} {total:>18.4f}")
print()
print(f"Diameter (extrema IVW):       {D_adopted:.3f} +/- {D_err:.3f} arcmin  (stat)")
print(f"Diameter (total budget):      {D_adopted:.3f} +/- {total:.3f} arcmin")
print()
print("Comparison values (see constants.py):")
print(f"  Optical nominal (aphelion):  {SOLAR_DIAMETER_ARCMIN_NOMINAL:.2f}'")
print(f"  Radio reference (17 GHz, Selhorst+04): 32.55'")
print(f"  Expected at 10 GHz: ~32.6'")


# Part V: Sunspot and Limb-Brightening Characterisation

## Physical basis

The 3-parameter envelope fit in Part III simultaneously recovers the limb-brightening coefficient $\varepsilon$ and the sunspot visibility floor $f$ for each frequency channel. This is a more principled decomposition than fitting a uniform disk and then attributing the null residuals post hoc, because:

1. **Limb brightening** modifies the entire envelope shape — not just the null depths — and the per-channel fit of $\varepsilon$ captures this globally rather than at a few discrete null positions.
2. **The sunspot floor $f$** is the residual after the limb-brightened disk model is subtracted. In the old null-residual approach, $f$ and $\varepsilon$ are degenerate at each null; here they are separated by their different $q$-dependence across the full envelope.

### Limb brightening at 10 GHz

At centimetre wavelengths, the quiet Sun's emission is thermal bremsstrahlung from the chromosphere. The chromospheric temperature increases with altitude, making the limb brighter than the disk centre — the opposite of optical limb darkening ([Kundu65]; [BBG98] §2; [Dulk85] §III). Fürst et al. (1979, A&A 76, 4) measured $\sim 10\text{-}25\%$ limb brightening at 10.7 GHz; Selhorst et al. (2004) found $\sim 30\text{-}40\%$ at 17 GHz.

### Sunspot visibility

A sunspot contributes an unresolved point-source visibility with constant amplitude but unknown phase; the quadrature addition $|V| = A\sqrt{V_{\mathrm{lb}}^2 + f^2}$ treats it as incoherent with the disk (see derivation in the title cell above). The fitted $f$ is the flux fraction of the unresolved component relative to the disk.


In [ ]:
# --- Sunspot and limb-brightening results from the 3-parameter fit ---
print("Per-channel envelope fit results (3-param: A, eps, f)")
print("=" * 55)

# Epsilon summary
eps_med = float(np.median(fit_eps))
eps_p16, eps_p84 = np.nanpercentile(fit_eps, [16, 84])
print(f"\nLimb-brightening coefficient epsilon:")
print(f"  median = {eps_med:.3f}")
print(f"  16-84 percentile = [{eps_p16:.3f}, {eps_p84:.3f}]")
print(f"  Literature expectation (10 GHz): ~0.1-0.25 (Fuerst+79)")
if eps_med > 1.5:
    print(f"  NOTE: median eps hits the upper bound (2.0) in many channels.")
    print(f"  This may indicate the quadratic profile is too restrictive,")
    print(f"  or that the data favours a stronger limb-brightening than")
    print(f"  the simple I(rho) = I0[1 + eps*(rho/R)^2] model can accommodate.")

# f (sunspot floor) summary
f_med = float(np.median(fit_f))
f_p16, f_p84 = np.nanpercentile(fit_f, [16, 84])
print(f"\nSunspot visibility floor f:")
print(f"  median = {f_med:.4f}")
print(f"  16-84 percentile = [{f_p16:.4f}, {f_p84:.4f}]")
print(f"  Interpretation: ~{f_med*100:.1f}% of the disk flux is from an")
print(f"  unresolved component (sunspot / active region).")

# Plot eps and f distributions
fig, axes = plt.subplots(1, 2, figsize=(TEXTWIDTH_IN, 3.0))

axes[0].hist(fit_eps, bins=30, color="C2", alpha=0.7, edgecolor="white")
axes[0].axvline(eps_med, color="C2", lw=LW_STANDARD, ls="--",
                label=f"median = {eps_med:.3f}")
axes[0].set_xlabel(r"$\varepsilon$")
axes[0].set_ylabel("Channels")
axes[0].set_title("Limb-brightening coefficient", fontsize=TICK_SIZE)
axes[0].legend(fontsize=TICK_SIZE - 1)

axes[1].hist(fit_f, bins=30, color="C3", alpha=0.7, edgecolor="white")
axes[1].axvline(f_med, color="C3", lw=LW_STANDARD, ls="--",
                label=f"median = {f_med:.4f}")
axes[1].set_xlabel("$f$ (sunspot floor)")
axes[1].set_ylabel("Channels")
axes[1].set_title("Sunspot visibility floor", fontsize=TICK_SIZE)
axes[1].legend(fontsize=TICK_SIZE - 1)

fig.tight_layout()
plt.show()


# Part VI: Discussion and Extensions

## 9.1 Systematic error budget context

The diameter fit from Part II returns a *statistical* uncertainty from the IVW of per-channel extrema positions. The envelope fit in Part III returns physically interpretable parameters ($\varepsilon$, $f$) but with $R$ fixed. The realistic total error bar is dominated by the baseline calibration. References to the underlying claims are in the table in §8.

## 9.2 Observation date and cross-check against the SDO/HMI sunspot catalogue

The chip metadata records the observation date in `unix_mid`. To validate the sunspot floor $f$ measured in Part III, the right cross-check is the **NOAA SWPC / SDO HMI sunspot catalogue** for that date:

- **NOAA SWPC Solar Region Summary**, https://www.swpc.noaa.gov/products/solar-region-summary
- **SDO HMI**, data archive at JSOC, https://jsoc.stanford.edu

In [ ]:
# --- Recover observation date for SDO/HMI cross-check ---
import datetime as _dt
t0 = float(unix_mid[0])
t1 = float(unix_mid[-1])
d0 = _dt.datetime.fromtimestamp(t0, tz=_dt.UTC)
d1 = _dt.datetime.fromtimestamp(t1, tz=_dt.UTC)
print(f"Observation start (UTC): {d0:%Y-%m-%d %H:%M:%S}")
print(f"Observation end   (UTC): {d1:%Y-%m-%d %H:%M:%S}")
print(f"Mean Sun declination:    {np.rad2deg(dec_rad_mean):.3f} deg")
print()
print("Cross-check the detected anomalies against:")
print(f"  https://jsoc.stanford.edu/ajax/lookdata.html?ds=hmi.M_45s"
      f"&op=exp_request&start={d0:%Y.%m.%d_%H:%M:%S_TAI}&"
      f"stop={d1:%Y.%m.%d_%H:%M:%S_TAI}")
print(f"  https://www.swpc.noaa.gov/products/solar-region-summary")
print()
print("Look for an active region on the visible solar disk near central")
print("meridian on this date. If the EW offset (delta_alpha) of any")
print("phase-localized spot above is consistent with a NOAA AR position,")
print("that is a successful interferometric detection.")


# Part VII: Summary of Results - three questions, three answers

The lab opened (notebook 01, "Scientific roadmap") with three questions:

> 1. How big is the Sun at 10 GHz?
> 2. Is there an active region on the disk today, and where is it?
> 3. Can we measure our own array geometry from the sky alone?

The next code cell prints the numerical results table. The markdown cell after it ties the three answers together into a single scientific story.

In [ ]:
# --- Final summary ---
print("=" * 64)
print("INTERFEROMETRIC SOLAR OBSERVATION - RESULTS SUMMARY")
print("=" * 64)

print(f"\nObserving frequency:     {band_center_hz/1e9:.4f} GHz")
print(f"Wavelength:              {C_LIGHT_MS / band_center_hz * 100:.2f} cm")
print(f"Sun declination:         {np.rad2deg(dec_rad_mean):.3f} deg")
print(f"Hour angle coverage:     {ha_deg_arr.min():.1f} to {ha_deg_arr.max():.1f} deg")
print(f"Total captures:          {len(unix_mid)}")
print(f"Observation chips:       {N_chips}")

print(f"\n--- Baseline (from notebook 04) ---")
print(f"  Adopted:  B_EW = {B_EW:.4f} m,  B_NS = {B_NS:.4f} m")

print(f"\n--- Solar Diameter ---")
print(f"  Geometric prior (Bessel extrema IVW):")
print(f"    D = {D_ivw:.3f} +/- {D_ivw_err:.3f} arcmin")
print(f"  Optical reference:       {SOLAR_DIAMETER_ARCMIN_NOMINAL:.1f} arcmin (photosphere)")
print(f"  Radio reference 17 GHz:  32.55 arcmin (Selhorst+04)")

print(f"\n--- Envelope Fit (limb-brightened disk + sunspot floor) ---")
print(f"  Limb-brightening eps:  median = {np.median(fit_eps):.3f}")
print(f"  Sunspot floor f:       median = {np.median(fit_f):.4f}")
print(f"  Converged channels:    {len(fit_results)} / {len(all_ch)}")


## Synthesis - the lab in three sentences

1. **Diameter first (question 1).** The adopted Bessel-envelope fit returns a radio-solar diameter that is larger than the optical $31.6'$ value and should be compared primarily to the cm-wavelength radio Sun, not the photosphere. Within the total error budget, agreement with the 17 GHz anchor of $32.55'$ and a slight upward shift at 10 GHz is the physically meaningful success criterion.

2. **Geometry second (question 3).** The fringe data recover an array geometry consistent with the lidar survey, and they do so much more precisely internally than the survey itself is known. For downstream quoting, however, the realistic uncertainty is still the **30 cm lidar floor**, not the much smaller formal fit covariance.

3. **Active-region extension (question 2).** Where the residual visibility at a Bessel null is significant, the joint amplitude+phase fit returns a flux fraction $f$ and an EW offset $\Delta\alpha$ from disk centre. NS position is not recoverable from this single EW baseline, so the sunspot result is necessarily one-dimensional.

### Bottom line

A two-element east-west interferometer at 10 GHz, recording for one afternoon, lets a student measure (i) **the chromospheric extension of the Sun**, (ii) **the EW offset of any active region on the disk**, and (iii) **the array's own geometry to within the 30 cm external survey floor**. Those three answers come from one calibrated dataset, one common fringe model, and one report-ready analysis chain.


## Extension: Fringe Model Comparison

With the baseline determined in notebook 04 and the solar diameter measured above, we reconstruct the expected fringe pattern — using the limb-brightened disk model from Part III — and compare to the data.

In [ ]:
# --- Fringe model comparison: data vs limb-brightened model ---
from utils import (
    point_source_visibility, FringeModelParams,
)

# Build the limb-brightened envelope model at the representative channel
k_show = k_mid
freq_hz_show = float(F_SKY_HZ[k_show])

# Get the fit result for this channel
fr = fit_results.get(k_show)
if fr is None:
    print(f"No fit result for channel {k_show}")
else:
    R_show = fr["R"]
    eps_show = fr["eps"]
    f_show = fr["f"]
    A_show = fr["A"]

    # Raw data
    obs_real = corr_dc[:, k_show].real
    obs_amp = np.abs(corr_dc[:, k_show])
    finite = np.isfinite(obs_real) & np.isfinite(ha_deg_arr)

    # Compute q for this channel
    q_show = compute_q(ha_rad_arr, dec_rad_mean, B_EW, B_NS, freq_hz_show)

    # Limb-brightened envelope
    v_lb = limb_brightened_visibility(q_show, R_show, eps_show)
    env_model = A_show * np.sqrt(v_lb**2 + f_show**2)

    # Sort by HA for plotting
    ha_sort = np.argsort(ha_deg_arr)

    fig, axes = plt.subplots(2, 1, figsize=(TEXTWIDTH_IN, 5.5), sharex=True,
                             gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.06})

    # Top: data + envelope
    axes[0].scatter(ha_deg_arr[finite], obs_amp[finite] / A_show,
                    s=0.3, alpha=0.05, color="C0", rasterized=True, label="$|V|$ data")
    axes[0].plot(ha_deg_arr[ha_sort], env_model[ha_sort] / A_show,
                 'C1-', lw=LW_STANDARD,
                 label=f"LB+spot model ($\\varepsilon$={eps_show:.2f}, f={f_show:.3f})")
    # Pure jinc for comparison
    from scipy.special import j1 as _J1
    x_jinc = 2 * np.pi * q_show * R_show
    jinc = np.where(x_jinc > 1e-8, np.abs(2 * _J1(x_jinc) / x_jinc), 1.0)
    axes[0].plot(ha_deg_arr[ha_sort], jinc[ha_sort],
                 'k--', lw=LW_LIGHT, alpha=0.4, label="Uniform disk jinc")
    axes[0].set_ylabel("Normalised $|V|$")
    axes[0].set_title(f"ch {k_show} ({F_SKY_GHZ[k_show]:.3f} GHz), "
                      f"D={np.rad2deg(2*R_show)*60:.1f}' (fixed)", fontsize=TICK_SIZE)
    axes[0].legend(fontsize=TICK_SIZE - 2, loc="upper right")
    axes[0].set_ylim(-0.05, 1.15)

    # Bottom: residual
    resid = obs_amp[finite] / A_show - env_model[finite] / A_show
    axes[1].scatter(ha_deg_arr[finite], resid,
                    s=0.3, alpha=0.05, color="C0", rasterized=True)
    axes[1].axhline(0, color="0.4", lw=LW_LIGHT, ls="--")
    rms = float(np.nanstd(resid))
    axes[1].axhspan(-rms, rms, alpha=0.1, color="C0",
                    label=f"$\\pm 1\\sigma$ ({rms:.3f})")
    axes[1].set_xlabel("Hour angle [deg]")
    axes[1].set_ylabel("Residual")
    axes[1].legend(fontsize=TICK_SIZE - 1)

    fig.tight_layout()
    plt.show()
    print(f"Residual RMS (normalised): {rms:.4f}")


## Extension: Consistency cross-check table

All measured quantities from this lab, collected in one place for easy comparison against priors, references, and each other. Agreements within quoted uncertainties are expected; persistent discrepancies flag unmodelled systematics.


In [ ]:
# --- Consistency cross-check table ---
from utils import (
    NOMINAL_B_EW_M, NOMINAL_B_EW_ERR_M,
    NOMINAL_B_NS_M, NOMINAL_B_NS_ERR_M,
    SOLAR_DIAMETER_ARCMIN_NOMINAL,
)

print("=" * 78)
print("  CONSISTENCY CROSS-CHECK TABLE")
print("=" * 78)

print("\n--- Baseline ---")
print(f"  {'Source':<35} {'b_ew [m]':>12} {'error [m]':>12}")
print("  " + "-" * 60)
print(f"  {'Lidar prior':<35} {NOMINAL_B_EW_M:>12.3f} {NOMINAL_B_EW_ERR_M:>12.3f}")
print(f"  {'Adopted (from nb 04)':<35} {B_EW:>12.4f} {B_EW_ERR:>12.4f}")

print(f"\n--- Solar diameter ---")
print(f"  {'Source':<35} {'diam [arcmin]':>14} {'error':>12}")
print("  " + "-" * 62)
print(f"  {'Lab manual nominal':<35} {SOLAR_DIAMETER_ARCMIN_NOMINAL:>14.2f} {'---':>12}")
print(f"  {'Radio ref (Selhorst04, 17 GHz)':<35} {'32.55':>14} {'0.05':>12}")
print(f"  {'Bessel extrema IVW':<35} {D_ivw:>14.3f} {D_ivw_err:>12.3f}")

print(f"\n--- Envelope fit parameters ---")
print(f"  {'Parameter':<35} {'median':>12} {'16-84 pct':>18}")
print("  " + "-" * 66)
print(f"  {'epsilon (limb brightening)':<35} {np.median(fit_eps):>12.3f} "
      f"[{np.percentile(fit_eps,16):.3f}, {np.percentile(fit_eps,84):.3f}]")
print(f"  {'f (sunspot floor)':<35} {np.median(fit_f):>12.4f} "
      f"[{np.percentile(fit_f,16):.4f}, {np.percentile(fit_f,84):.4f}]")

print(f"\n--- Amplitude systematics ---")
print(f"  Chip gains = {chip_gains}")
_dc_path = Path("_dc_correction_alpha.pkl")
if _dc_path.exists():
    import pickle as _p
    with open(_dc_path, "rb") as _ff:
        _dca = float(_p.load(_ff)["alpha"])
    print(f"  DC correction alpha = {_dca:.4f} ({(_dca-1)*100:+.1f}% bias, corrected)")

print("\n" + "=" * 78)
